# QSP and Q# profiling comparison

Run all cells in order using the repository's `.venv` kernel. Native compilation requires a C++ compiler and `cargo build --release -p qep_interop` from the repository root.

Each benchmark executes the same program on 50 qubits: H and MResetZ on qubit 0, one million iterations of H and T on every qubit, then Reset and M on qubit 0. The first measurement supplies the loop's starting index, keeping the loop dynamic in QIR. Every run verifies **100,000,004 quantum operations** and the per-operation counts.

The primary **ns/op** figure includes compilation, startup, execution, transport (where applicable), and result loading, but excludes application construction and count validation. Q# uses the result adapter to normalize stored measurements and result reads to the same quantum operations as native QSP; its overhead is included.

## 1. Native QSP over pipes

In [1]:
import subprocess
from time import perf_counter

from qdk import qsharp
from qdk.qre import ProfilerResult
from qdk.qre.application import QSharpApplication, QspApplication
from qdk.qre.profiler import QREProfiler

EXPECTED_CALLS = {
    "h": 50_000_001, "t": 50_000_000, "mz_reset": 1, "reset": 1, "mz": 1
}
EXPECTED_COUNT = sum(EXPECTED_CALLS.values())


def report(label: str, result: ProfilerResult, elapsed: float) -> None:
    """Verify the shared workload and report end-to-end time per quantum operation."""
    count = result.totals.total_operations_count
    assert count == EXPECTED_COUNT
    assert result.totals.calls_per_operation == EXPECTED_CALLS
    assert result.totals.logical_qubits == 50
    print(f"{label}: {1e9 * elapsed / count:.1f} ns/op (end-to-end)")
    print(f"{count:,} ops in {elapsed:.3f}s; {count / elapsed / 1e6:.3f} million ops/s")


CPP_WORKLOAD = """
#include "qep_interop.h"

// Execute the shared workload and check measurement replies.
int run_workload() {
    qdk_h(0);
    const size_t start = qdk_mz_reset(0);
    if (start > 1) return 1;
    for (size_t i = start; i < start + 1000000; ++i) {
        for (size_t j = 0; j < 50; ++j) {
            qdk_h(j);
            qdk_t(j);
        }
    }
    qdk_reset(0);
    return qdk_mz(0) > 1;
}
"""
pipe_app = QspApplication.for_cpp_code(CPP_WORKLOAD + """
// Send the workload to the external profiler over pipes.
int main() {
    return run_workload();
}
""")
profiler = QREProfiler.counting()
started = perf_counter()
result = profiler.profile(pipe_app)
elapsed = perf_counter() - started
report("Native QSP over pipes", result, elapsed)

Native QSP over pipes: 88.5 ns/op (end-to-end)
100,000,004 ops in 8.852s; 11.297 million ops/s


## 2. Native QSP with an in-process profiler

In [2]:
in_process_app = QspApplication.for_cpp_code(CPP_WORKLOAD + """
// Run with a local counting profiler and save its result.
int main() {
    qdk_init("counting_profiler");
    const int status = run_workload();
    qdk_save_profile("profile.json");
    return status;
}
""")
started = perf_counter()
executable = in_process_app.compile()
compiled = perf_counter()
subprocess.run([str(executable)], cwd=executable.parent, check=True)
finished = perf_counter()
result = ProfilerResult.from_json((executable.parent / "profile.json").read_text(encoding="utf-8"))
elapsed = perf_counter() - started
report("Native QSP in-process", result, elapsed)
print(f"Execution only (excluding compilation and result loading): {1e9 * (finished - compiled) / EXPECTED_COUNT:.1f} ns/op")

Profile saved to profile.json
Native QSP in-process: 28.8 ns/op (end-to-end)
100,000,004 ops in 2.884s; 34.669 million ops/s
Execution only (excluding compilation and result loading): 26.8 ns/op


## 3. Q# interpreter (`use_qir=False`)

A measurement-dependent `while` loop preserves the native loop bounds without unrolling one million iterations during QIR compilation.

In [3]:
qsharp.init(target_profile=qsharp.TargetProfile.Adaptive)
qsharp_app = QSharpApplication("""
{
    use qs = Qubit[50];
    H(qs[0]);
    let start = MResetZ(qs[0]) == One ? 1 | 0;
    mutable i = start;
    while i < start + 1000000 {
        for j in 0..49 {
            H(qs[j]);
            T(qs[j]);
        }
        set i += 1;
    }
    Reset(qs[0]);
    M(qs[0])
}
""")
profiler = QREProfiler.counting()
profiler.profiler.add_adapter("result_adapter")
started = perf_counter()
result = profiler.profile(qsharp_app, use_qir=False)
elapsed = perf_counter() - started
report("Q# interpreter (use_qir=False)", result, elapsed)

Q# interpreter (use_qir=False): 369.6 ns/op (end-to-end)
100,000,004 ops in 36.963s; 2.705 million ops/s


## 4. Q# via QIR (`use_qir=True`)

Reuse exactly the same `QSharpApplication` with a fresh counting profiler.

In [4]:
profiler = QREProfiler.counting()
profiler.profiler.add_adapter("result_adapter")
started = perf_counter()
result = profiler.profile(qsharp_app, use_qir=True)
elapsed = perf_counter() - started
report("Q# via QIR (use_qir=True)", result, elapsed)

Q# via QIR (use_qir=True): 85.6 ns/op (end-to-end)
100,000,004 ops in 8.562s; 11.680 million ops/s
